<a href="https://colab.research.google.com/github/aniketkumar112sonu-ui/fraud-detection-risk-engine/blob/main/notebook_01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv("/content/paysim dataset.csv")

print(df.shape)

(6362620, 11)


In [3]:
print(df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [4]:
print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB
None


In [6]:
print(df.isnull().sum())

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [7]:
df[df.isnull().any(axis=1)]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud


In [13]:
df = df.dropna()

In [14]:
print(df.isnull().sum().sum())

0


In [8]:
print(df["isFraud"].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [9]:
print(df["isFraud"].value_counts(normalize=True) * 100)

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [10]:
print(df["type"].value_counts())

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [15]:
print("\nFraud by transaction type:")
print(pd.crosstab(df["type"], df["isFraud"]))


Fraud by transaction type:
isFraud         0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097


In [16]:
fraud_rate = df.groupby("type")["isFraud"].mean() * 100
print(fraud_rate.sort_values(ascending=False))

type
TRANSFER    0.768799
CASH_OUT    0.183955
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64


In [17]:
print(df.groupby("isFraud")["amount"].agg(["count", "mean", "median", "max"]))

           count          mean     median          max
isFraud                                               
0        6354407  1.781970e+05   74684.72  92445516.64
1           8213  1.467967e+06  441423.44  10000000.00


In [18]:
df["orig_balance_change"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["dest_balance_change"] = df["newbalanceDest"] - df["oldbalanceDest"]

In [19]:
print(df.groupby("isFraud")[[
    "orig_balance_change",
    "dest_balance_change"
]].mean())


         orig_balance_change  dest_balance_change
isFraud                                          
0              -2.314152e+04        123504.809994
1               1.457275e+06        735457.998071


In [20]:
df["orig_error"] = df["amount"] - df["orig_balance_change"]

print("\nOrigin balance error:")
print(df.groupby("isFraud")["orig_error"].mean())


Origin balance error:
isFraud
0    201338.558109
1     10692.325265
Name: orig_error, dtype: float64


In [21]:
# Create time-based features
df["hour"] = df["step"] % 24
df["day"] = df["step"] // 24
df["is_night"] = ((df["hour"] >= 0) & (df["hour"] < 6)).astype(int)

# Compare fraud rate by hour
hourly_fraud = df.groupby("hour")["isFraud"].mean() * 100

print("Fraud rate by hour:")
print(hourly_fraud.round(3))

print("\nFraud rate at night:")
print(df.groupby("is_night")["isFraud"].mean().mul(100).round(3))

Fraud rate by hour:
hour
0      0.419
1      1.320
2      4.125
3     16.243
4     22.079
5     22.303
6     10.468
7      3.649
8      1.367
9      0.120
10     0.088
11     0.073
12     0.070
13     0.074
14     0.080
15     0.082
16     0.078
17     0.080
18     0.059
19     0.053
20     0.061
21     0.140
22     0.180
23     0.229
Name: isFraud, dtype: float64

Fraud rate at night:
is_night
0    0.099
1    1.773
Name: isFraud, dtype: float64


In [22]:
print("isFlaggedFraud distribution:")
print(df["isFlaggedFraud"].value_counts())

print("\nFraud rate by isFlaggedFraud:")
print(df.groupby("isFlaggedFraud")["isFraud"].mean().mul(100).round(3))

print("\nFraud transactions that were flagged:")
print(pd.crosstab(df["isFraud"], df["isFlaggedFraud"]))

isFlaggedFraud distribution:
isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

Fraud rate by isFlaggedFraud:
isFlaggedFraud
0      0.129
1    100.000
Name: isFraud, dtype: float64

Fraud transactions that were flagged:
isFlaggedFraud        0   1
isFraud                    
0               6354407   0
1                  8197  16


In [23]:
print("Duplicate rows:", df.duplicated().sum())

print("\nDuplicate Transaction IDs:", df["nameOrig"].duplicated().sum())

print("\nUnique transaction types:", df["type"].nunique())
print(df["type"].unique())

Duplicate rows: 0

Duplicate Transaction IDs: 9313

Unique transaction types: 5
['PAYMENT' 'TRANSFER' 'CASH_OUT' 'DEBIT' 'CASH_IN']


In [24]:
print(df.describe().T)

                         count          mean           std          min  \
step                 6362620.0  2.433972e+02  1.423320e+02         1.00   
amount               6362620.0  1.798619e+05  6.038582e+05         0.00   
oldbalanceOrg        6362620.0  8.338831e+05  2.888243e+06         0.00   
newbalanceOrig       6362620.0  8.551137e+05  2.924049e+06         0.00   
oldbalanceDest       6362620.0  1.100702e+06  3.399180e+06         0.00   
newbalanceDest       6362620.0  1.224996e+06  3.674129e+06         0.00   
isFraud              6362620.0  1.290820e-03  3.590480e-02         0.00   
isFlaggedFraud       6362620.0  2.514687e-06  1.585775e-03         0.00   
orig_balance_change  6362620.0 -2.123056e+04  1.466433e+05  -1915267.90   
dest_balance_change  6362620.0  1.242947e+05  8.129391e+05 -13060826.21   
orig_error           6362620.0  2.010925e+05  6.066505e+05        -0.01   
hour                 6362620.0  1.532145e+01  4.321799e+00         0.00   
day                  6362

In [26]:
import numpy as np
df["amount_log"] = np.log1p(df["amount"])

In [27]:
print(df[[
    "amount",
    "orig_balance_change",
    "dest_balance_change",
    "orig_error",
    "hour",
    "day",
    "is_night",
    "amount_log"
]].head())

     amount  orig_balance_change  dest_balance_change    orig_error  hour  \
0   9839.64              9839.64                  0.0 -1.455192e-11     1   
1   1864.28              1864.28                  0.0  1.136868e-12     1   
2    181.00               181.00                  0.0  0.000000e+00     1   
3    181.00               181.00             -21182.0  0.000000e+00     1   
4  11668.14             11668.14                  0.0  0.000000e+00     1   

   day  is_night  amount_log  
0    0         1    9.194276  
1    0         1    7.531166  
2    0         1    5.204007  
3    0         1    5.204007  
4    0         1    9.364703  
